# Holdings And Metadata

Explore ETF metadata stored in ETF Terminal and optionally pull live holdings from FMP.

In [ ]:
import pandas as pd

from config import FMP_API_KEY, FMP_BASE_URL
from db.connection import get_engine
from services.market.fmp_client import FMPClient
from stores.market import MetadataStore, SecurityStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
TICKER = "LQD"

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
metadata_store = MetadataStore(engine)
security_store = SecurityStore(engine)
fmp_client = FMPClient(api_key=FMP_API_KEY, base_url=FMP_BASE_URL)

In [ ]:
metadata = metadata_store.get_ticker_metadata(TICKER) or {}
pd.Series(metadata).sort_index()

In [ ]:
securities = security_store.list_active_securities()
securities.head(10)

In [ ]:
# Live FMP holdings pull (not persisted to your DB by this notebook)
holdings = fmp_client.get_etf_holdings(TICKER)
holdings_df = pd.DataFrame(holdings)
holdings_df.head(15)

In [ ]:
preferred_columns = [
    col for col in ["asset", "name", "symbol", "weightPercentage", "sharesNumber", "marketValue"]
    if col in holdings_df.columns
]

if "weightPercentage" in holdings_df.columns:
    holdings_df["weightPercentage"] = pd.to_numeric(holdings_df["weightPercentage"], errors="coerce")
    holdings_df = holdings_df.sort_values("weightPercentage", ascending=False)

holdings_df[preferred_columns].head(20)